In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations



import pickle

from pathlib import Path

from typing import Any



import numpy as np

import pandas as pd



RESULTS_DIR = Path(".").resolve() / "results"

BINARY_PKL = RESULTS_DIR / "results_ensure_binary.pkl"

MULTICLASS_PKL = RESULTS_DIR / "results_ensure_multiclass.pkl"

REGRESSION_PKL = RESULTS_DIR / "results_ensure_regression.pkl"



pd.set_option("display.max_rows", 200)

pd.set_option("display.max_columns", 200)


In [3]:
def _load(path: Path) -> dict[str, Any] | None:

    if not path.exists():

        print(f"Missing: {path}")

        return None

    with path.open("rb") as f:

        return pickle.load(f)





binary = _load(BINARY_PKL)

multiclass = _load(MULTICLASS_PKL)

regression = _load(REGRESSION_PKL)


In [4]:
CANON_COLUMNS = [

    "Total",

    "CoFa",

    "CoPo",

    "SeFa",

    "SePo",

    "SuFa",

    "SuPo",

    "Ens",

    "Time",

]





def _counts_row(counts_mean: dict[str, float], time_mean_seconds: float) -> dict[str, float]:

    return {

        "Total": float(counts_mean.get("total", np.nan)),

        "CoFa": float(counts_mean.get("counterfactual", np.nan)),

        "CoPo": float(counts_mean.get("counterpotential", np.nan)),

        "SeFa": float(counts_mean.get("semifactual", np.nan)),

        "SePo": float(counts_mean.get("semipotential", np.nan)),

        "SuFa": float(counts_mean.get("superfactual", np.nan)),

        "SuPo": float(counts_mean.get("superpotential", np.nan)),

        "Ens": float(counts_mean.get("ensured", np.nan)),

        "Time": float(time_mean_seconds),

    }





def extract_counts_tables(result_blob: dict[str, Any], *, task: str, setting: str | None = None) -> dict[str, pd.DataFrame]:

    """Return {dataset_name: df(index=cal_size x stage)}."""



    tables: dict[str, pd.DataFrame] = {}

    for ds_name, ds_res in result_blob.get("results", {}).items():

        if task == "regression":

            if setting is None:

                raise ValueError("setting is required for regression")

            if setting == "plain":

                ds_setting = ds_res["plain"]

            else:

                ds_setting = ds_res["probabilistic"][setting]

            by_cal = ds_setting["by_calibration_size"]

        else:

            by_cal = ds_res["by_calibration_size"]



        rows: dict[str, dict[str, float]] = {}

        for cal_size in sorted(by_cal.keys(), reverse=False):

            explore = by_cal[cal_size]["explore"]

            conj = by_cal[cal_size]["conjugate"]

            rows[f"{cal_size} (s)"] = _counts_row(

                explore["counts_mean"], explore.get("time_mean_seconds", 0.0)

            )

            rows[f"{cal_size} (c)"] = _counts_row(

                conj["counts_mean"], conj.get("time_mean_seconds", 0.0)

            )



        df = pd.DataFrame.from_dict(rows, orient="index")

        tables[ds_name] = df[CANON_COLUMNS]



    return tables





def aggregate_tables_mean_std(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:

    if not tables:

        return pd.DataFrame(columns=CANON_COLUMNS)



    any_df = next(iter(tables.values()))

    out = pd.DataFrame(index=any_df.index, columns=any_df.columns, dtype=object)

    for idx in out.index:

        for col in out.columns:

            vals = np.asarray([df.loc[idx, col] for df in tables.values()], dtype=float)

            vals = vals[~np.isnan(vals)]

            if vals.size == 0:

                out.loc[idx, col] = ""

            else:

                out.loc[idx, col] = f"{np.mean(vals):.2f} ({np.std(vals):.2f})"

    return out





def extract_ranking_table(result_blob: dict[str, Any], *, task: str, setting: str | None = None) -> pd.DataFrame:

    """Long-form ranking table across datasets.



    Columns include:

    - ensured_reachability_rate

    - per-weight: score~uncertainty corr, score~prediction corr

    - top1 ensured/pareto rates

    - boundary_match_rate (only meaningful at w in {-1,0,1})

    """



    rows: list[dict[str, Any]] = []

    for ds_name, ds_res in result_blob.get("results", {}).items():

        if task == "regression":

            if setting is None:

                raise ValueError("setting is required for regression")

            if setting == "plain":

                per_cal = ds_res["plain"]["by_calibration_size"]

            else:

                per_cal = ds_res["probabilistic"][setting]["by_calibration_size"]

        else:

            per_cal = ds_res["by_calibration_size"]



        for cal_size, stages in per_cal.items():

            for stage in ("explore", "conjugate"):

                rv = stages[stage]["ranking_validation"]

                ensured_reach = float(rv.get("ensured_reachability_rate", np.nan))

                per_w = rv.get("per_w", {})

                for w_str, vals in per_w.items():

                    rows.append(

                        {

                            "dataset": ds_name,

                            "cal_size": int(cal_size),

                            "stage": stage,

                            "w": float(w_str),

                            "ensured_reachability_rate": ensured_reach,

                            "spearman_score_uncertainty": vals.get("spearman_score_uncertainty"),

                            "spearman_score_prediction": vals.get("spearman_score_prediction"),

                            "top1_ensured_rate": vals.get("top1_ensured_rate"),

                            "top1_pareto_rate": vals.get("top1_pareto_rate"),

                            "boundary_match_rate": vals.get("boundary_match_rate"),

                        }

                    )

    df = pd.DataFrame(rows)

    if df.empty:

        return df

    return df.sort_values(["cal_size", "stage", "w", "dataset"]).reset_index(drop=True)





def aggregate_ranking(df: pd.DataFrame) -> pd.DataFrame:

    if df.empty:

        return df

    metric_cols = [

        "ensured_reachability_rate",

        "spearman_score_uncertainty",

        "spearman_score_prediction",

        "top1_ensured_rate",

        "top1_pareto_rate",

        "boundary_match_rate",

    ]

    agg = (

        df.groupby(["cal_size", "stage", "w"], as_index=False)[metric_cols]

        .agg(["mean", "std"])

        .reset_index()

    )

    # Flatten multiindex columns

    agg.columns = [

        col[0] if col[1] == "" else f"{col[0]}_{col[1]}" for col in agg.columns.to_flat_index()

    ]

    return agg





def monotonicity_over_w(agg: pd.DataFrame) -> pd.DataFrame:

    """Compute simple monotonicity checks: corr(w, metric_mean) per cal/stage."""

    if agg.empty:

        return agg



    out_rows: list[dict[str, Any]] = []

    for (cal_size, stage), g in agg.groupby(["cal_size", "stage"]):

        w = g["w"].to_numpy(dtype=float)



        def _corr(col: str) -> float:

            y = g[col].to_numpy(dtype=float)

            if len(w) < 2:

                return float("nan")

            return float(pd.Series(w).corr(pd.Series(y), method="spearman"))



        out_rows.append(

            {

                "cal_size": int(cal_size),

                "stage": stage,

                "corr_w_top1_ensured_rate": _corr("top1_ensured_rate_mean"),

                "corr_w_top1_pareto_rate": _corr("top1_pareto_rate_mean"),

                "corr_w_score_pred_corr": _corr("spearman_score_prediction_mean"),

                "corr_w_score_unc_corr": _corr("spearman_score_uncertainty_mean"),

            }

        )



    return pd.DataFrame(out_rows).sort_values(["cal_size", "stage"]).reset_index(drop=True)


In [5]:
# Counts/time tables (mean ± std across datasets)



if binary is not None:

    display(aggregate_tables_mean_std(extract_counts_tables(binary, task="binary")))



if multiclass is not None:

    display(aggregate_tables_mean_std(extract_counts_tables(multiclass, task="multiclass")))



if regression is not None:

    display(aggregate_tables_mean_std(extract_counts_tables(regression, task="regression", setting="plain")))

    display(aggregate_tables_mean_std(extract_counts_tables(regression, task="regression", setting="p25")))

    display(aggregate_tables_mean_std(extract_counts_tables(regression, task="regression", setting="p50")))

    display(aggregate_tables_mean_std(extract_counts_tables(regression, task="regression", setting="p75")))


,Total,CoFa,CoPo,SeFa,SePo,SuFa,SuPo,Ens,Time
2 (s),0.23 (0.00),0.23 (0.00),0.00 (0.00),0.23 (0.00),0.00 (0.00),0.23 (0.00),0.00 (0.00),0.23 (0.00),0.00 (0.00)
2 (c),0.23 (0.00),0.00 (0.00),0.23 (0.00),0.00 (0.00),0.23 (0.00),0.00 (0.00),0.13 (0.00),0.10 (0.00),0.00 (0.00)


,Total,CoFa,CoPo,SeFa,SePo,SuFa,SuPo,Ens,Time
47 (s),5.28 (0.00),5.28 (0.00),0.00 (0.00),5.28 (0.00),0.00 (0.00),5.28 (0.00),0.00 (0.00),5.28 (0.00),0.00 (0.00)
47 (c),11.28 (0.00),2.77 (0.00),3.39 (0.00),3.89 (0.00),3.39 (0.00),1.23 (0.00),0.20 (0.00),2.89 (0.00),0.03 (0.00)


,Total,CoFa,CoPo,SeFa,SePo,SuFa,SuPo,Ens,Time
30 (s),15.10 (0.00),15.10 (0.00),0.00 (0.00),15.10 (0.00),0.00 (0.00),15.10 (0.00),0.00 (0.00),15.10 (0.00),0.01 (0.00)
30 (c),31.10 (0.00),0.00 (0.00),10.50 (0.00),0.00 (0.00),10.50 (0.00),7.20 (0.00),13.40 (0.00),20.90 (0.00),0.07 (0.00)


,Total,CoFa,CoPo,SeFa,SePo,SuFa,SuPo,Ens,Time
30 (s),14.10 (0.00),14.10 (0.00),0.00 (0.00),14.10 (0.00),0.00 (0.00),14.10 (0.00),0.00 (0.00),14.10 (0.00),0.01 (0.00)
30 (c),30.50 (0.00),10.20 (0.00),6.60 (0.00),5.70 (0.00),6.60 (0.00),8.00 (0.00),0.30 (0.00),14.50 (0.00),0.09 (0.00)


,Total,CoFa,CoPo,SeFa,SePo,SuFa,SuPo,Ens,Time
30 (s),11.70 (0.00),11.70 (0.00),0.00 (0.00),11.70 (0.00),0.00 (0.00),11.70 (0.00),0.00 (0.00),11.70 (0.00),0.01 (0.00)
30 (c),26.20 (0.00),6.80 (0.00),4.40 (0.00),6.60 (0.00),4.40 (0.00),8.40 (0.00),0.10 (0.00),18.40 (0.00),0.11 (0.00)


,Total,CoFa,CoPo,SeFa,SePo,SuFa,SuPo,Ens,Time
30 (s),12.00 (0.00),12.00 (0.00),0.00 (0.00),12.00 (0.00),0.00 (0.00),12.00 (0.00),0.00 (0.00),12.00 (0.00),0.01 (0.00)
30 (c),25.30 (0.00),8.10 (0.00),7.00 (0.00),5.50 (0.00),7.00 (0.00),4.70 (0.00),1.30 (0.00),11.60 (0.00),0.07 (0.00)


In [6]:
# Ranking metric validation + Pareto/ensured checks



def show_ranking(blob: dict[str, Any] | None, *, task: str, setting: str | None = None, title: str) -> None:

    if blob is None:

        return

    print(title)

    df = extract_ranking_table(blob, task=task, setting=setting)

    agg = aggregate_ranking(df)

    display(agg)

    display(monotonicity_over_w(agg))





show_ranking(binary, task="binary", title="Binary classification")

show_ranking(multiclass, task="multiclass", title="Multiclass classification")



show_ranking(regression, task="regression", setting="plain", title="Regression (plain)")

show_ranking(regression, task="regression", setting="p25", title="Regression (probabilistic, p25)")

show_ranking(regression, task="regression", setting="p50", title="Regression (probabilistic, p50)")

show_ranking(regression, task="regression", setting="p75", title="Regression (probabilistic, p75)")


Binary classification


,index,cal_size,stage,w,ensured_reachability_rate_mean,ensured_reachability_rate_std,spearman_score_uncertainty_mean,spearman_score_uncertainty_std,spearman_score_prediction_mean,spearman_score_prediction_std,top1_ensured_rate_mean,top1_ensured_rate_std,top1_pareto_rate_mean,top1_pareto_rate_std,boundary_match_rate_mean,boundary_match_rate_std
0,0,2,conjugate,-1.0,0.10,NaN,NaN,NaN,NaN,NaN,0.434783,NaN,1.0,NaN,1.0,NaN
1,1,2,conjugate,-0.5,0.10,NaN,NaN,NaN,NaN,NaN,0.434783,NaN,1.0,NaN,NaN,NaN
2,2,2,conjugate,0.0,0.10,NaN,NaN,NaN,NaN,NaN,0.434783,NaN,1.0,NaN,1.0,NaN
3,3,2,conjugate,0.5,0.10,NaN,NaN,NaN,NaN,NaN,0.434783,NaN,1.0,NaN,NaN,NaN
4,4,2,conjugate,1.0,0.10,NaN,NaN,NaN,NaN,NaN,0.434783,NaN,1.0,NaN,1.0,NaN
5,5,2,explore,-1.0,0.23,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,1.0,NaN,1.0,NaN
6,6,2,explore,-0.5,0.23,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,1.0,NaN,NaN,NaN
7,7,2,explore,0.0,0.23,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,1.0,NaN,1.0,NaN
8,8,2,explore,0.5,0.23,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,1.0,NaN,NaN,NaN
9,9,2,explore,1.0,0.23,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,1.0,NaN,1.0,NaN


c:\Users\loftuw\AppData\Local\r-miniconda\Lib\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,cal_size,stage,corr_w_top1_ensured_rate,corr_w_top1_pareto_rate,corr_w_score_pred_corr,corr_w_score_unc_corr
0,2,conjugate,NaN,NaN,NaN,NaN
1,2,explore,NaN,NaN,NaN,NaN


Multiclass classification


...

,cal_size,stage,corr_w_top1_ensured_rate,corr_w_top1_pareto_rate,corr_w_score_pred_corr,corr_w_score_unc_corr
0,30,conjugate,0.866025,NaN,1.0,-0.6
1,30,explore,NaN,NaN,1.0,-0.6


Regression (probabilistic, p75)


,index,cal_size,stage,w,ensured_reachability_rate_mean,ensured_reachability_rate_std,spearman_score_uncertainty_mean,spearman_score_uncertainty_std,spearman_score_prediction_mean,spearman_score_prediction_std,top1_ensured_rate_mean,top1_ensured_rate_std,top1_pareto_rate_mean,top1_pareto_rate_std,boundary_match_rate_mean,boundary_match_rate_std
0,0,30,conjugate,-1.0,0.9,NaN,0.540898,NaN,-1.000000,NaN,0.5,NaN,1.0,NaN,1.0,NaN
1,1,30,conjugate,-0.5,0.9,NaN,-0.036490,NaN,-0.684319,NaN,0.5,NaN,1.0,NaN,NaN,NaN
2,2,30,conjugate,0.0,0.9,NaN,-0.999135,NaN,0.540630,NaN,0.9,NaN,1.0,NaN,1.0,NaN
3,3,30,conjugate,0.5,0.9,NaN,-0.976871,NaN,0.576082,NaN,0.9,NaN,1.0,NaN,NaN,NaN
4,4,30,conjugate,1.0,0.9,NaN,-0.540898,NaN,1.000000,NaN,0.9,NaN,0.8,NaN,1.0,NaN
5,5,30,explore,-1.0,1.0,NaN,0.578990,NaN,-1.000000,NaN,1.0,NaN,1.0,NaN,1.0,NaN
6,6,30,explore,-0.5,1.0,NaN,-0.052715,NaN,-0.571640,NaN,1.0,NaN,1.0,NaN,NaN,NaN
7,7,30,explore,0.0,1.0,NaN,-1.000000,NaN,0.578990,NaN,1.0,NaN,1.0,NaN,1.0,NaN
8,8,30,explore,0.5,1.0,NaN,-0.954202,NaN,0.624882,NaN,1.0,NaN,1.0,NaN,NaN,NaN
9,9,30,explore,1.0,1.0,NaN,-0.578990,NaN,1.000000,NaN,1.0,NaN,1.0,NaN,1.0,NaN


c:\Users\loftuw\AppData\Local\r-miniconda\Lib\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,cal_size,stage,corr_w_top1_ensured_rate,corr_w_top1_pareto_rate,corr_w_score_pred_corr,corr_w_score_unc_corr
0,30,conjugate,0.866025,-0.707107,1.0,-0.6
1,30,explore,NaN,NaN,1.0,-0.6
